### 16. NetMiko

Netmiko is a Python Library used for network automation.

It helps us connect network devices like:

* Cisco Routers
* Cisco Switches
* Juniper Devices
* Fortinet Firewalls
* HP/Aruba Switches
* Linux Server(via- SSH)
* Mikrotik

Netmiko works main using SSH(Secure Shell)

✅ Why Netmiko is Used?

Without Netmiko, network engineers configure devices manully by:

* Opening PuTTY/Secure CRT
* Logging into device
* typing commands one by one

With Netmiko, we can automate:
* ✅ Show commands
* ✅ Configuration commands
* ✅ Backup Configuration
* ✅ Multiple devices configuration
* ✅ network monitoring
* ✅ log collection

✅ Netmiko Architecture
Netmiko internally uses:

#### Paramiko
A Python SSHClient library.

Netmiko is built on top of Paramiko and adds:

* device type support
* auto login prompts
* enable mode handling
* send_config_set function
* session logging


### Basic Netmiko workflow

The normal workflow in Netmiko is:

1. Import ConnectHandler
2. Create device dictionary.
3. Connect to device
4. Send show commands or config commands
5. Disconnect session

## Basic Example Code

Connect and Run Show command

In [2]:
from netmiko import ConnectHandler

In [3]:
pip show netmiko

Name: netmiko
Version: 4.6.0
Summary: Multi-vendor library to simplify legacy CLI connections to network devices
Home-page: https://github.com/ktbyers/netmiko
Author: Kirk Byers
Author-email: ktbyers@twb-tech.com
License: MIT
Location: C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages
Requires: ntc-templates, paramiko, pyserial, pyyaml, rich, ruamel.yaml, scp, textfsm
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [36]:
device={
    "device_type":"cisco_ios",
    "host":"192.168.0.254",
    "username":"cisco",
    "password":"cisco@123",
    "secret":"cisco@123",
    "port":22
}

connection = ConnectHandler(**device,timeout=10,banner_timeout=10,auth_timeout=10)
print(connection)

# Enter enable mode (privileged mode)
connection.enable()

# Run to show command
output = connection.send_command("show ip interface brief")

print(output)


Interface                  IP-Address      OK? Method Status                Protocol
GigabitEthernet0/0         192.168.0.254   YES NVRAM  up                    up      
GigabitEthernet0/1         unassigned      YES NVRAM  administratively down down    
Serial0/3/0                unassigned      YES NVRAM  administratively down down    
Serial0/3/1                unassigned      YES NVRAM  administratively down down    


In [35]:
from netmiko import ConnectHandler
from netmiko.exceptions import NetmikoTimeoutException, NetmikoAuthenticationException

device = {
    "device_type": "cisco_ios",
    "host": "192.168.0.254",
    "username": "cisco",
    "password": "cisco@123",
    "secret": "cisco@123",
    "port": 22,
}

try:
    print("Connecting to device...")

    connection = ConnectHandler(
        **device,
        timeout=20,
        banner_timeout=20,
        auth_timeout=20,
    )

    print("Connected successfully ✅")

    # Enter enable mode
    connection.enable()

    # Run multiple commands safely
    commands = [
        "terminal length 0",
        "show ip interface brief",
        "show version"
    ]

    for cmd in commands:
        print(f"\n===== {cmd} =====")
        output = connection.send_command(cmd)
        print(output)

    # Close connection
    connection.disconnect()
    print("\nDisconnected successfully 🔚")

except NetmikoTimeoutException:
    print("❌ ERROR: Device not reachable (Check IP / Network / SSH)")

except NetmikoAuthenticationException:
    print("❌ ERROR: Authentication failed (Check username/password)")

except Exception as e:
    print(f"❌ Unexpected Error: {e}")

Connecting to device...
Connected successfully ✅

===== terminal length 0 =====


===== show ip interface brief =====
Interface                  IP-Address      OK? Method Status                Protocol
GigabitEthernet0/0         192.168.0.254   YES NVRAM  up                    up      
GigabitEthernet0/1         unassigned      YES NVRAM  administratively down down    
Serial0/3/0                unassigned      YES NVRAM  administratively down down    
Serial0/3/1                unassigned      YES NVRAM  administratively down down    

===== show version =====
Cisco IOS Software, 2800 Software (C2800NM-ADVIPSERVICESK9-M), Version 12.4(24)T1, RELEASE SOFTWARE (fc3)
Technical Support: http://www.cisco.com/techsupport
Copyright (c) 1986-2009 by Cisco Systems, Inc.
Compiled Fri 19-Jun-09 15:13 by prod_rel_team

ROM: System Bootstrap, Version 12.4(13r)T11, RELEASE SOFTWARE (fc1)

R1 uptime is 1 hour, 20 minutes
System returned to ROM by power-on
System image file is "flash:c2800nm-advipse

In [34]:
from netmiko import ConnectHandler
from netmiko.exceptions import (
    NetmikoTimeoutException,
    NetmikoAuthenticationException,
)
import socket

device = {
    "device_type": "cisco_ios",
    "host": "192.168.0.254",
    "username": "cisco",
    "password": "cisco@123",
    "secret": "cisco@123",   # must match 'enable secret' on device
    "port": 22,
    "timeout": 20,
    "banner_timeout": 20,
    "auth_timeout": 20,
    "fast_cli": False,       # more reliable on some devices
}

def check_reachability(host, port=22, timeout=5):
    """Quick TCP check before Netmiko tries to connect."""
    try:
        sock = socket.create_connection((host, port), timeout=timeout)
        sock.close()
        return True
    except Exception:
        return False

def main():
    print("🔎 Checking reachability...")
    if not check_reachability(device["host"], device["port"]):
        print("❌ Cannot reach device on TCP 22 (IP/SSH issue).")
        return

    try:
        print("🔐 Connecting via Netmiko...")
        conn = ConnectHandler(**device)
        print("✅ Connected")

        

        # Enter enable mode (uses 'secret')
        if not conn.check_enable_mode():
            conn.enable()
        print("🔓 In enable mode")

        # Good practice for Cisco paging
        conn.send_command("terminal length 0")

        commands = [
            "show ip interface brief",
            "show version",
            "show running-config | section vty",
        ]

        for cmd in commands:
            print(f"\n===== {cmd} =====")
            output = conn.send_command(cmd)
            print(output)

        conn.disconnect()
        print("\n🔚 Disconnected cleanly")

    except NetmikoTimeoutException:
        print("❌ Timeout: Device not reachable / SSH not open")
    except NetmikoAuthenticationException:
        print("❌ Auth failed: Check username/password/secret")
    except Exception as e:
        print(f"❌ Unexpected error: {e}")

if __name__ == "__main__":
    main()

🔎 Checking reachability...
🔐 Connecting via Netmiko...
✅ Connected
🔓 In enable mode

===== show ip interface brief =====
Interface                  IP-Address      OK? Method Status                Protocol
GigabitEthernet0/0         192.168.0.254   YES NVRAM  up                    up      
GigabitEthernet0/1         unassigned      YES NVRAM  administratively down down    
Serial0/3/0                unassigned      YES NVRAM  administratively down down    
Serial0/3/1                unassigned      YES NVRAM  administratively down down    

===== show version =====
Cisco IOS Software, 2800 Software (C2800NM-ADVIPSERVICESK9-M), Version 12.4(24)T1, RELEASE SOFTWARE (fc3)
Technical Support: http://www.cisco.com/techsupport
Copyright (c) 1986-2009 by Cisco Systems, Inc.
Compiled Fri 19-Jun-09 15:13 by prod_rel_team

ROM: System Bootstrap, Version 12.4(13r)T11, RELEASE SOFTWARE (fc1)

R1 uptime is 1 hour, 20 minutes
System returned to ROM by power-on
System image file is "flash:c2800nm-advi

In [33]:
from netmiko import ConnectHandler

device = {
    "device_type": "cisco_ios",
    "host": "192.168.0.254",
    "username": "cisco",
    "password": "cisco@123",
    "secret": "cisco@123",   # must match enable secret
    "port": 22,
}

conn = ConnectHandler(**device)

# Enter enable mode
conn.enable()

print("✅ Enable mode entered")

output = conn.send_command("show ip interface brief")
print(output)

conn.disconnect()

✅ Enable mode entered
Interface                  IP-Address      OK? Method Status                Protocol
GigabitEthernet0/0         192.168.0.254   YES NVRAM  up                    up      
GigabitEthernet0/1         unassigned      YES NVRAM  administratively down down    
Serial0/3/0                unassigned      YES NVRAM  administratively down down    
Serial0/3/1                unassigned      YES NVRAM  administratively down down    
